In [ ]:
# .venv/bin/pip install trl peft accelerate bitsandbytes datasets

In [ ]:
import importlib, sys

packages = [
    "torch", "transformers", "peft", "trl",
    "bitsandbytes", "accelerate", "datasets",
    "vllm", "sympy", "numpy", "tokenizers",
]

print(f"Python: {sys.version}\n" + "-" * 50)
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg:<20} {getattr(mod, '__version__', 'unknown')}")
    except ImportError:
        print(f"{pkg:<20} NOT INSTALLED")

# Also check CUDA
try:
    import torch
    print(f"\n{'CUDA available':<20} {torch.cuda.is_available()}")
    print(f"{'CUDA version':<20} {torch.version.cuda}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            print(f"  GPU {i}: {props.name}  |  {props.total_memory / 1e9:.1f} GB  |  SM {props.major}.{props.minor}")
except ImportError:
    pass

## Dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("open-r1/OpenR1-Math-220k", name="default")
train_ds = ds["train"]

print(f"Loaded {len(train_ds)} examples")
# Expected: ~93,000 examples

SYSTEM_PROMPT = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

def format_example(example):
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": example["problem"]},
            {"role": "assistant", "content": example["solution"]},
        ]
    }

train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,        # nested quantization → saves ~0.4 GB
    bnb_4bit_quant_type="nf4",             # NormalFloat4 — optimal for normally distributed weights
    bnb_4bit_compute_dtype=torch.float16,  # A30 lacks BF16 tensor cores — use fp16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False          # Required for gradient checkpointing
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"        # Required for SFT packing

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)  # Casts LayerNorm to fp32, enables grad checkpointing

lora_config = LoraConfig(
    r=32,                          # Rank — higher = more capacity, more memory
    lora_alpha=16,                 # Scaling factor: effective LR ≈ lora_alpha / r
    target_modules=[               # Qwen3 attention + FFN projections
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~1–3% of total params trainable

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./checkpoints/sft_qlora",
    num_train_epochs=1,              # 1 epoch recommended — long R1 traces, large dataset
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,      # Effective batch size = 16
    gradient_checkpointing=True,         # Saves ~30% VRAM at cost of ~20% speed
    optim="paged_adamw_32bit",           # QLoRA-recommended optimizer
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    fp16=True,                           # A30 lacks BF16 tensor cores — use fp16 instead
    logging_steps=50,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    report_to="none",
    dataloader_num_workers=4,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    tokenizer=tokenizer,
    max_seq_length=4096,                  # Truncate long CoT solutions
    dataset_text_field=None,              # Use "messages" field with chat template
    packing=True,                          # Pack short examples into full-length sequences → faster training
)

trainer.train()

In [ ]:
# Must reload base model in fp16/bf16 (not 4-bit) for merging
from peft import PeftModel
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,   # A30: fp16
    device_map="auto",
    trust_remote_code=True,
)
peft_model = PeftModel.from_pretrained(base_model, "./models/sft_qlora_adapter")
merged_model = peft_model.merge_and_unload()

merged_model.save_pretrained("./models/sft_merged", safe_serialization=True)
tokenizer.save_pretrained("./models/sft_merged")